# LeetCode 1078 — Occurrences After Bigram bằng Python

**Link đề:** https://leetcode.com/problems/occurrences-after-bigram/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def findOcurrences(self, text: str, first: str, second: str) -> list[str]:
        ...
```

(Chú ý: tên method trên LeetCode viết **`findOcurrences`** — thiếu một chữ `c`. Phải giữ đúng như vậy khi nộp.)

## Mục tiêu học
1. Hiểu **bigram** (cặp hai từ liền nhau) và yêu cầu "từ thứ ba đứng ngay sau cặp `first second`".
2. Dùng `str.split()` tách câu thành list từ; duyệt bằng chỉ số `range(len(words) - 2)` mà không vượt biên.
3. Biết cách **cửa sổ trượt cỡ 3** (sliding window) bằng `zip(words, words[1:], words[2:])`.
4. Viết bản **một lượt, không tạo list** (giữ 2 từ trước đó) — tư duy xử lý luồng dữ liệu.
5. Trace, chứng minh tính đúng, Time/Space, các lỗi hay gặp (chồng nhau, so khớp chuỗi con…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — từ nào đứng ngay sau "first second"?

Cho hai từ `first`, `second`. Xét mọi chỗ trong `text` có dạng **`first second third`** — tức `second` đứng **ngay sau** `first`, và `third` đứng **ngay sau** `second`. Trả về list tất cả các từ `third` (theo thứ tự xuất hiện).

**Ví dụ 1:** `text = "alice is a good girl she is a good student"`, `first = "a"`, `second = "good"` → `["girl", "student"]`

**Ví dụ 2:** `text = "we will we will rock you"`, `first = "we"`, `second = "will"` → `["we", "rock"]`
(chú ý: từ `"we"` thứ hai **vừa là** `third` của lần 1, **vừa là** `first` của lần 2 — các lần xuất hiện được phép **chồng nhau**.)

**Constraints:**
- `1 <= text.length <= 1000`, gồm chữ thường và dấu cách.
- Các từ cách nhau **đúng một** dấu cách; không có dấu cách ở đầu/cuối.
- `1 <= first.length, second.length <= 10`, chữ thường.

In [1]:
text = "alice is a good girl she is a good student"
words = text.split()
for i, w in enumerate(words):
    print(i, w)

0 alice
1 is
2 a
3 good
4 girl
5 she
6 is
7 a
8 good
9 student


Nhìn danh sách trên: `a` ở chỉ số 2, `good` ở 3 → từ ở 4 là `girl`. `a` ở 7, `good` ở 8 → từ ở 9 là `student`.

## 2. Nền tảng Python cần biết

### 2.1. `str.split()` — tách chuỗi thành list từ

- `s.split()` (không tham số) tách theo **mọi khoảng trắng**, tự bỏ khoảng trắng thừa.
- `s.split(" ")` tách theo **đúng** một dấu cách; nếu có 2 dấu cách liền sẽ sinh chuỗi rỗng `''`.

Đề đảm bảo đúng một dấu cách, nên hai cách cho cùng kết quả; `split()` an toàn hơn.

In [2]:
print("we will rock".split())
print("we  will".split())        # ['we', 'will']
print("we  will".split(" "))     # ['we', '', 'will']

['we', 'will', 'rock']
['we', 'will']
['we', '', 'will']


### 2.2. Duyệt bộ ba liên tiếp bằng chỉ số

Bộ ba bắt đầu ở `i` là `words[i], words[i+1], words[i+2]`. Chỉ số lớn nhất được đọc là `i + 2`, phải `<= len(words) - 1` ⇒ `i <= len(words) - 3` ⇒ `for i in range(len(words) - 2)`.

Nếu `len(words) < 3` thì `range` âm/0 → **không lặp lần nào**, tự an toàn.

In [3]:
words = ["w0", "w1", "w2", "w3", "w4"]
for i in range(len(words) - 2):
    print(i, (words[i], words[i + 1], words[i + 2]))
print("list ngắn:", list(range(len(["a", "b"]) - 2)))   # [] -> không lặp

0 ('w0', 'w1', 'w2')
1 ('w1', 'w2', 'w3')
2 ('w2', 'w3', 'w4')
list ngắn: []


### 2.3. `zip` nhiều list lệch nhau — cửa sổ trượt

`words[1:]` là list bỏ phần tử đầu; `words[2:]` bỏ 2 phần tử đầu. `zip` ghép theo vị trí và **dừng ở list ngắn nhất**:

```text
words     : w0 w1 w2 w3 w4
words[1:] : w1 w2 w3 w4
words[2:] : w2 w3 w4
zip       : (w0,w1,w2) (w1,w2,w3) (w2,w3,w4)
```

In [4]:
words = ["w0", "w1", "w2", "w3", "w4"]
for triple in zip(words, words[1:], words[2:]):
    print(triple)

('w0', 'w1', 'w2')
('w1', 'w2', 'w3')
('w2', 'w3', 'w4')


### 2.4. So sánh chuỗi bằng `==`

`"a" == "a"` là `True`; `"a" == "an"` là `False`. So sánh **cả từ**, khác với kiểm tra **chuỗi con** `"a" in "an"` (`True`). Bài này cần so cả từ.

In [5]:
print("a" == "a", "a" == "an", "a" in "an")

True False True


## 3. Cách 1 — Tách từ + duyệt chỉ số (lời giải trực giác)

1. `words = text.split()`.
2. Với `i` trong `range(len(words) - 2)`: nếu `words[i] == first` **và** `words[i+1] == second` thì thêm `words[i+2]`.
3. Trả về list kết quả.

In [6]:
def find_occurrences_index(text: str, first: str, second: str) -> list[str]:
    words = text.split()
    res = []
    for i in range(len(words) - 2):
        if words[i] == first and words[i + 1] == second:
            res.append(words[i + 2])
    return res

print(find_occurrences_index("alice is a good girl she is a good student", "a", "good"))
print(find_occurrences_index("we will we will rock you", "we", "will"))

['girl', 'student']
['we', 'rock']


## 4. Trace cách 1 với `"we will we will rock you"`, `first="we"`, `second="will"`

`words = ["we","will","we","will","rock","you"]`, `len = 6` → `i` chạy `0..3`.

| `i` | `words[i]` | `words[i+1]` | Khớp? | `words[i+2]` | `res` sau |
|:---:|---|---|:---:|---|---|
| 0 | we | will | **có** | we | `["we"]` |
| 1 | will | we | không | — | `["we"]` |
| 2 | we | will | **có** | rock | `["we","rock"]` |
| 3 | will | rock | không | — | `["we","rock"]` |

In [7]:
def trace_find_occurrences(text: str, first: str, second: str) -> list[str]:
    words = text.split()
    print("words =", words)
    res = []
    for i in range(len(words) - 2):
        match = words[i] == first and words[i + 1] == second
        if match:
            res.append(words[i + 2])
        print(f"i={i}: ({words[i]!r}, {words[i + 1]!r}) khớp={match!s:5} -> res={res}")
    return res

trace_find_occurrences("we will we will rock you", "we", "will")

words = ['we', 'will', 'we', 'will', 'rock', 'you']
i=0: ('we', 'will') khớp=True  -> res=['we']
i=1: ('will', 'we') khớp=False -> res=['we']
i=2: ('we', 'will') khớp=True  -> res=['we', 'rock']
i=3: ('will', 'rock') khớp=False -> res=['we', 'rock']


['we', 'rock']

## 5. Cách 2 — Cửa sổ trượt bằng `zip` + list comprehension

Viết gọn cách 1: đi qua từng bộ ba `(a, b, c)`, giữ `c` khi `a == first and b == second`.

**List comprehension** `[biểu_thức for ... in ... if điều_kiện]` tạo list mới chỉ với các phần tử thỏa điều kiện.

In [8]:
def find_occurrences_zip(text: str, first: str, second: str) -> list[str]:
    words = text.split()
    return [c for a, b, c in zip(words, words[1:], words[2:]) if a == first and b == second]

print(find_occurrences_zip("alice is a good girl she is a good student", "a", "good"))
print(find_occurrences_zip("we will we will rock you", "we", "will"))

['girl', 'student']
['we', 'rock']


## 6. Nút thắt & insight: có cần lưu cả list từ không?

Cách 1, 2 tạo list `words` (và cách 2 còn tạo thêm 2 bản sao `words[1:]`, `words[2:]`) → bộ nhớ `O(n)`.

Nếu `text` là một **luồng dữ liệu rất dài** (đọc từng từ một), ta chỉ cần nhớ **hai từ vừa đọc**:

- `prev2` = từ cách đây 2 bước, `prev1` = từ ngay trước.
- Đọc từ mới `w`: nếu `prev2 == first and prev1 == second` thì `w` là một `third`.
- Trượt cửa sổ: `prev2, prev1 = prev1, w`.

Đây là tư duy **cửa sổ trượt với trạng thái cố định** — bộ nhớ chỉ `O(1)` từ (ngoài kết quả).

In [9]:
# Từng bước trượt cửa sổ 2 từ
prev2, prev1 = None, None
for w in ["we", "will", "we"]:
    print(f"trước: prev2={prev2!r}, prev1={prev1!r}, đọc w={w!r}")
    prev2, prev1 = prev1, w
print("sau cùng:", prev2, prev1)

trước: prev2=None, prev1=None, đọc w='we'
trước: prev2=None, prev1='we', đọc w='will'
trước: prev2='we', prev1='will', đọc w='we'
sau cùng: will we


## 7. Cách 3 (lời giải chính cho phỏng vấn) — Một lượt, giữ 2 từ trước

Ghép từng đoạn:

**Đoạn 1:** `prev2 = prev1 = None`, `res = []`.

**Đoạn 2:** đọc lần lượt từng từ `w`. Để thật sự không tạo list, có thể dùng **generator** sinh từ ra từng cái một (mục 7.1); ở đây trước hết dùng `text.split()` cho dễ hiểu.

**Đoạn 3:** kiểm tra `prev2 == first and prev1 == second` → `res.append(w)`.

**Đoạn 4:** trượt `prev2, prev1 = prev1, w`.

In [10]:
def find_occurrences_stream(text: str, first: str, second: str) -> list[str]:
    prev2 = prev1 = None
    res = []
    for w in text.split():
        if prev2 == first and prev1 == second:
            res.append(w)
        prev2, prev1 = prev1, w
    return res

print(find_occurrences_stream("alice is a good girl she is a good student", "a", "good"))
print(find_occurrences_stream("we will we will rock you", "we", "will"))

['girl', 'student']
['we', 'rock']


In [11]:
def trace_stream(text: str, first: str, second: str) -> list[str]:
    prev2 = prev1 = None
    res = []
    for w in text.split():
        hit = prev2 == first and prev1 == second
        if hit:
            res.append(w)
        print(f"w={w!r:9} | (prev2, prev1)=({prev2!r}, {prev1!r}) khớp={hit!s:5} | res={res}")
        prev2, prev1 = prev1, w
    return res

trace_stream("alice is a good girl she is a good student", "a", "good")

w='alice'   | (prev2, prev1)=(None, None) khớp=False | res=[]
w='is'      | (prev2, prev1)=(None, 'alice') khớp=False | res=[]
w='a'       | (prev2, prev1)=('alice', 'is') khớp=False | res=[]
w='good'    | (prev2, prev1)=('is', 'a') khớp=False | res=[]
w='girl'    | (prev2, prev1)=('a', 'good') khớp=True  | res=['girl']
w='she'     | (prev2, prev1)=('good', 'girl') khớp=False | res=['girl']
w='is'      | (prev2, prev1)=('girl', 'she') khớp=False | res=['girl']
w='a'       | (prev2, prev1)=('she', 'is') khớp=False | res=['girl']
w='good'    | (prev2, prev1)=('is', 'a') khớp=False | res=['girl']
w='student' | (prev2, prev1)=('a', 'good') khớp=True  | res=['girl', 'student']


['girl', 'student']

### 7.1. Tách từ "lười" bằng generator (không tạo list)

`yield` biến hàm thành **generator**: mỗi lần vòng `for` cần một từ, hàm chạy tiếp tới `yield` kế tiếp. Ta tự quét ký tự, gom thành từ khi gặp dấu cách.

In [12]:
def iter_words(text: str):
    start = 0
    for i, ch in enumerate(text):
        if ch == " ":
            yield text[start:i]
            start = i + 1
    yield text[start:]                 # từ cuối cùng (không có dấu cách sau)

gen = iter_words("we will rock")
print(type(gen).__name__)
print(next(gen), next(gen), next(gen))

def find_occurrences_lazy(text: str, first: str, second: str) -> list[str]:
    prev2 = prev1 = None
    res = []
    for w in iter_words(text):
        if prev2 == first and prev1 == second:
            res.append(w)
        prev2, prev1 = prev1, w
    return res

print(find_occurrences_lazy("we will we will rock you", "we", "will"))

generator
we will rock
['we', 'rock']


## 8. Code nộp LeetCode

### Bản chỉ số (dễ đọc nhất)

In [13]:
class Solution:
    def findOcurrences(self, text: str, first: str, second: str) -> list[str]:
        words = text.split()
        res = []
        for i in range(len(words) - 2):
            if words[i] == first and words[i + 1] == second:
                res.append(words[i + 2])
        return res

sol = Solution()
assert sol.findOcurrences("alice is a good girl she is a good student", "a", "good") == ["girl", "student"]
assert sol.findOcurrences("we will we will rock you", "we", "will") == ["we", "rock"]
print("Solution PASS")

Solution PASS


### Bản một lượt `prev2/prev1`

In [14]:
class SolutionStream:
    def findOcurrences(self, text: str, first: str, second: str) -> list[str]:
        prev2 = prev1 = None
        res = []
        for w in text.split():
            if prev2 == first and prev1 == second:
                res.append(w)
            prev2, prev1 = prev1, w
        return res

assert SolutionStream().findOcurrences("we will we will rock you", "we", "will") == ["we", "rock"]
print("SolutionStream PASS")

SolutionStream PASS


## 9. Vì sao đúng?

**Cách 1:** mỗi lần xuất hiện `first second third` ứng với **đúng một** chỉ số `i` (vị trí của `first`), với `0 <= i <= len(words) - 3`. Vòng `for` xét **mọi** `i` như vậy theo thứ tự tăng ⇒ không bỏ sót, không trùng, đúng thứ tự. Các lần chồng nhau (như ví dụ 2) là các `i` khác nhau nên đều được xét.

**Cách 3 — invariant:** ngay trước khi xử lý từ thứ `k` (`w = words[k]`), `prev1 = words[k-1]` và `prev2 = words[k-2]` (hoặc `None` nếu chưa tồn tại). Vậy điều kiện `prev2 == first and prev1 == second` đúng ⇔ bộ ba kết thúc tại `k` khớp mẫu. `None` không bằng chuỗi nào nên hai từ đầu tiên không bao giờ bị ghi nhầm.

## 10. Độ phức tạp

Gọi `n = len(text)` (số ký tự), `m` = số từ.

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1. split + chỉ số | `O(n)` | `O(n)` (list từ) |
| 2. zip 3 lát cắt | `O(n)` | `O(n)` (list từ + 2 bản sao lát cắt) |
| 3. prev2/prev1 + split | `O(n)` | `O(n)` vì `split` vẫn tạo list |
| 3'. prev2/prev1 + generator | `O(n)` | `O(L)` với `L` = độ dài từ dài nhất |

So sánh chuỗi `==` tốn tối đa `O(độ dài từ)`, cộng lại vẫn `O(n)`.

## 11. Bộ test

Nhóm test:
1. Hai ví dụ đề.
2. Ít hơn 3 từ → `[]`.
3. `first == second`: `"a a a a"`, `first = second = "a"` → `["a", "a"]` (chồng nhau dày đặc).
4. Mẫu ở cuối câu, không có `third` → không ghi.
5. Từ chứa `first` như chuỗi con (`"an"` chứa `"a"`) → không được nhầm.
6. Đối chiếu ngẫu nhiên các cách.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS -> {actual}")

tests = [
    ("đề 1", "alice is a good girl she is a good student", "a", "good", ["girl", "student"]),
    ("đề 2", "we will we will rock you", "we", "will", ["we", "rock"]),
    ("1 từ", "hello", "hello", "world", []),
    ("2 từ khớp nhưng không có third", "a good", "a", "good", []),
    ("first == second", "a a a a", "a", "a", ["a", "a"]),
    ("mẫu ở cuối", "x y z a good", "a", "good", []),
    ("chuỗi con không tính", "an goods a good day", "a", "good", ["day"]),
    ("không khớp", "one two three", "two", "one", []),
]
for name, text, f, s, exp in tests:
    check(name, find_occurrences_index(text, f, s), exp)
    assert find_occurrences_zip(text, f, s) == exp
    assert find_occurrences_stream(text, f, s) == exp
    assert find_occurrences_lazy(text, f, s) == exp

đề 1                       PASS -> ['girl', 'student']
đề 2                       PASS -> ['we', 'rock']
1 từ                       PASS -> []
2 từ khớp nhưng không có third PASS -> []
first == second            PASS -> ['a', 'a']
mẫu ở cuối                 PASS -> []
chuỗi con không tính       PASS -> ['day']
không khớp                 PASS -> []


In [16]:
import random
random.seed(1078)
vocab = ["a", "b", "ab", "ba"]
for _ in range(3000):
    text = " ".join(random.choice(vocab) for _ in range(random.randint(1, 12)))
    f, s = random.choice(vocab), random.choice(vocab)
    r = [find_occurrences_index(text, f, s), find_occurrences_zip(text, f, s),
         find_occurrences_stream(text, f, s), find_occurrences_lazy(text, f, s)]
    assert all(x == r[0] for x in r), (text, f, s, r)
print("3000 test ngẫu nhiên: 4 cách khớp nhau")

3000 test ngẫu nhiên: 4 cách khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Duyệt `range(len(words))` rồi đọc `words[i + 2]` → `IndexError`

In [17]:
def wrong_range(text, first, second):
    words = text.split()
    res = []
    for i in range(len(words)):
        if words[i] == first and words[i + 1] == second:
            res.append(words[i + 2])
    return res

try:
    wrong_range("x a good", "a", "good")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 2: Nhảy qua bộ ba sau khi khớp (`i += 3`) → mất lần xuất hiện chồng nhau

In [18]:
def wrong_skip(text, first, second):
    words = text.split()
    res, i = [], 0
    while i < len(words) - 2:
        if words[i] == first and words[i + 1] == second:
            res.append(words[i + 2])
            i += 3                      # nhảy quá xa
        else:
            i += 1
    return res

print("sai :", wrong_skip("we will we will rock you", "we", "will"))
print("đúng:", find_occurrences_index("we will we will rock you", "we", "will"))

sai : ['we']
đúng: ['we', 'rock']


### Lỗi 3: Tìm chuỗi con `first + " " + second` bằng `find` trên `text`

Chuỗi `"a good "` có xuất hiện trong `"ka good day"` (đuôi `a` của từ `ka` + dấu cách + `good`), dù từ `a` **không hề** có ở đó. Tìm theo chuỗi con dễ khớp **nửa từ**. Tách từ rồi so cả từ thì an toàn.

In [19]:
def wrong_substring(text, first, second):
    res = []
    pattern = first + " " + second + " "
    i = text.find(pattern)
    while i != -1:
        rest = text[i + len(pattern):]
        res.append(rest.split()[0])
        i = text.find(pattern, i + 1)
    return res

t = "ka good day x a good night"
print("sai :", wrong_substring(t, "a", "good"))
print("đúng:", find_occurrences_index(t, "a", "good"))

sai : ['day', 'night']
đúng: ['night']


### Lỗi 4: Viết sai tên method khi nộp

LeetCode gọi `findOcurrences` (một chữ `c`). Viết `findOccurrences` sẽ báo lỗi "has no attribute". Giữ **đúng skeleton** của đề.

### Lỗi 5: Trượt cửa sổ sai thứ tự

```python
prev1 = w
prev2 = prev1     # sai: prev2 thành w luôn
```
Dùng gán đồng thời `prev2, prev1 = prev1, w`.

## 13. Mẹo nhận dạng pattern

- Đề nói về các **phần tử liền kề cố định k** (cặp, bộ ba…) → cửa sổ trượt cỡ `k`: chỉ số `range(n - k + 1)`, hoặc `zip(a, a[1:], ..., a[k-1:])`, hoặc giữ `k - 1` phần tử trước.
- Văn bản → **tách từ trước**, so sánh **cả từ** — tránh so chuỗi con.
- "Có thể chồng nhau" → bước nhảy **1**, không nhảy qua cả mẫu.

| Bài liên quan | Ý tưởng |
|---|---|
| 1550. Three Consecutive Odds | cửa sổ 3 phần tử |
| 643. Maximum Average Subarray I | cửa sổ cỡ k cộng dồn |
| 884. Uncommon Words from Two Sentences | split + đếm |
| 1455. Check If a Word Occurs As a Prefix | split + startswith |

## 14. Biến thể

### Biến thể A — Mẫu dài `k` từ bất kỳ (n-gram), trả về từ đứng sau

In [20]:
def words_after_ngram(text: str, pattern: list[str]) -> list[str]:
    words = text.split()
    k = len(pattern)
    return [words[i + k] for i in range(len(words) - k) if words[i:i + k] == pattern]

print(words_after_ngram("we will we will rock you", ["we", "will"]))
print(words_after_ngram("a b c a b c a b d", ["a", "b", "c"]))

['we', 'rock']
['a', 'a']


### Biến thể B — Đếm tần suất mọi bigram (nền tảng mô hình ngôn ngữ)

In [21]:
from collections import Counter

def bigram_counts(text: str) -> Counter:
    words = text.split()
    return Counter(zip(words, words[1:]))

print(bigram_counts("we will we will rock you").most_common(3))

[(('we', 'will'), 2), (('will', 'we'), 1), (('will', 'rock'), 1)]


## 15. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `text = "to be or not to be or to be"`, `first = "to"`, `second = "be"`. Kết quả là gì?

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_find_occurrences` theo TODO (cách chỉ số).

**Bài 3 (viết lại):** `ex_find_stream` — viết lại bằng `prev2/prev1`, không dùng chỉ số.

**Bài 4 (biến thể):** `ex_words_before(text, second, third)` — trả về mọi từ `first` đứng **ngay trước** cặp `second third`.

**Bài 5 (mini challenge):** `ex_predict_next(text, word)` — trả về từ **hay xuất hiện nhất** ngay sau `word` (hòa thì lấy từ nhỏ nhất theo thứ tự từ điển); không có thì `""`.

In [22]:
def ex_find_occurrences(text: str, first: str, second: str) -> list[str]:
    words = text.split()
    res = []
    # TODO 1: for i in range(...)  (chú ý biên!)
    # TODO 2: nếu words[i] == first và words[i + 1] == second -> thêm words[i + 2]
    return res

# assert ex_find_occurrences("to be or not to be or to be", "to", "be") == ["or", "or"]

In [23]:
def ex_find_stream(text: str, first: str, second: str) -> list[str]:
    # TODO: prev2 = prev1 = None; duyệt từng từ; trượt bằng gán đồng thời
    pass

# assert ex_find_stream("we will we will rock you", "we", "will") == ["we", "rock"]

In [24]:
def ex_words_before(text: str, second: str, third: str) -> list[str]:
    # TODO: bộ ba (a, b, c): giữ a khi b == second và c == third
    pass

# assert ex_words_before("alice is a good girl she is a good student", "good", "girl") == ["a"]

In [25]:
def ex_predict_next(text: str, word: str) -> str:
    # TODO: đếm các từ đứng sau `word` bằng Counter; chọn theo (-số lần, từ)
    pass

# assert ex_predict_next("we will we will rock you", "will") == "rock"
# assert ex_predict_next("a b a c a b", "a") == "b"
# assert ex_predict_next("a b", "z") == ""

## 16. Gợi ý 3 mức

**Bài 1:** (1) Tách từ, đánh số. (2) `to be` ở (0,1), (4,5), (7,8). (3) Sau (0,1) là `or`; sau (4,5) là `or`; (7,8) ở cuối, không có `third` → `["or", "or"]`.

**Bài 2:** (1) Như mục 3. (2) `range(len(words) - 2)`. (3) `res.append(words[i + 2])`.

**Bài 3:** (1) Như mục 7. (2) Kiểm tra trước, trượt sau. (3) `prev2, prev1 = prev1, w`.

**Bài 4:** (1) Bộ ba bằng `zip`. (2) Điều kiện đặt lên `b`, `c`. (3) `[a for a, b, c in zip(...) if b == second and c == third]`.

**Bài 5:** (1) Đếm các cặp `(w, next)` với `w == word`. (2) `Counter(nxt for w, nxt in zip(words, words[1:]) if w == word)`. (3) `min(cnt, key=lambda x: (-cnt[x], x))`, rỗng thì `""`.

## 17. Đáp án tham khảo

Tự làm trước rồi mới xem. Bài 1: `["or", "or"]`.

In [26]:
def ans_find_occurrences(text, first, second):
    words = text.split()
    res = []
    for i in range(len(words) - 2):
        if words[i] == first and words[i + 1] == second:
            res.append(words[i + 2])
    return res

def ans_find_stream(text, first, second):
    prev2 = prev1 = None
    res = []
    for w in text.split():
        if prev2 == first and prev1 == second:
            res.append(w)
        prev2, prev1 = prev1, w
    return res

def ans_words_before(text, second, third):
    words = text.split()
    return [a for a, b, c in zip(words, words[1:], words[2:]) if b == second and c == third]

def ans_predict_next(text, word):
    words = text.split()
    cnt = Counter(nxt for w, nxt in zip(words, words[1:]) if w == word)
    if not cnt:
        return ""
    return min(cnt, key=lambda x: (-cnt[x], x))

assert ans_find_occurrences("to be or not to be or to be", "to", "be") == ["or", "or"]
assert ans_find_stream("we will we will rock you", "we", "will") == ["we", "rock"]
assert ans_words_before("alice is a good girl she is a good student", "good", "girl") == ["a"]
assert ans_predict_next("we will we will rock you", "will") == "rock"
assert ans_predict_next("a b a c a b", "a") == "b"
assert ans_predict_next("a b", "z") == ""
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findOcurrences(self, text: str, first: str, second: str) -> list[str]:
        words = text.split()
        return [c for a, b, c in zip(words, words[1:], words[2:])
                if a == first and b == second]
```

**Khuôn cửa sổ cỡ k:** `for i in range(len(a) - k + 1): window = a[i:i + k]`.

**Checklist:**
- [ ] Tên method đúng `findOcurrences` (một chữ `c`).
- [ ] Biên vòng lặp `range(len(words) - 2)` — không `IndexError`.
- [ ] Bước nhảy 1 để bắt lần xuất hiện **chồng nhau**.
- [ ] So sánh **cả từ** sau khi `split`, không tìm chuỗi con.
- [ ] Test `"we will we will rock you"` → `["we","rock"]`, `"a a a a"` → `["a","a"]`.
- [ ] Time `O(n)`, Space `O(n)` (hoặc `O(1)` phụ với bản luồng).